# DeepX Arabic ABSA — Preprocessing Pipeline

This notebook covers every preprocessing step needed before model training.  
Each step is motivated by a concrete finding from EDA.

**Pipeline overview:**
1. Load & parse structured columns
2. Deduplicate
3. Normalize `business_category` → macro-category
4. Clean Arabic review text
5. Handle class imbalance
6. Create two training formats: (a) flattened per-aspect rows, (b) per-review multi-label
7. Train / validation split
8. Save

## 0. Install & Imports

In [2]:
# Uncomment if running fresh in Colab
# !pip install pandas openpyxl scikit-learn arabert camel-tools -q

import pandas as pd
import numpy as np
import ast
import re
import json
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
ASPECT_TAXONOMY = ['food', 'service', 'price', 'cleanliness',
                   'delivery', 'ambiance', 'app_experience', 'general', 'none']
SENTIMENT_CLASSES = ['positive', 'negative', 'neutral']

df = pd.read_excel('DeepX_train.xlsx')   # adjust path as needed
print(f'Loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head(2)

Loaded: 1971 rows, 9 columns


,review_id,review_text,star_rating,date,business_name,business_category,platform,aspects,aspect_sentiments
0,7238,لا يوجد الدفع بالبطاقه عند الاستلام,3,2026-03-08 00:00:00,Noon,ecommerce,play_store,"[""app_experience"", ""delivery""]","{""app_experience"": ""negative"", ""delivery"": ""ne..."
1,1036,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,5,قبل يومين (2),ممشي مصر Mawlana Cafe,كافيه,google_maps,"[""cleanliness"", ""ambiance"", ""service""]","{""cleanliness"": ""positive"", ""ambiance"": ""posit..."


---
## Step 1 — Parse Structured String Columns
**Why:** `aspects` and `aspect_sentiments` are stored as Python-literal strings,  
not actual lists/dicts. Every downstream operation requires parsed objects.

In [3]:
def safe_parse(val, fallback):
    """Parse a Python-literal string safely."""
    try:
        return ast.literal_eval(val)
    except Exception:
        return fallback

df['aspects_list']    = df['aspects'].apply(lambda x: safe_parse(x, []))
df['sentiments_dict'] = df['aspect_sentiments'].apply(lambda x: safe_parse(x, {}))

# Sanity check: every aspect in the list should appear as a key in sentiments_dict
mismatch = df.apply(
    lambda r: set(r['aspects_list']) != set(r['sentiments_dict'].keys()), axis=1
).sum()
print(f'Rows where aspects_list ≠ sentiments_dict keys: {mismatch}')

# Validate against taxonomy
unknown_aspects = set()
for lst in df['aspects_list']:
    for a in lst:
        if a not in ASPECT_TAXONOMY:
            unknown_aspects.add(a)
print(f'Unknown aspects outside taxonomy: {unknown_aspects}')

unknown_sentiments = set()
for d in df['sentiments_dict']:
    for s in d.values():
        if s not in SENTIMENT_CLASSES:
            unknown_sentiments.add(s)
print(f'Unknown sentiment values: {unknown_sentiments}')

Rows where aspects_list ≠ sentiments_dict keys: 0
Unknown aspects outside taxonomy: set()
Unknown sentiment values: set()


---
## Step 2 — Deduplicate
**Why:** 88 rows (4.5%) share the same review text. Some have conflicting labels.  
Duplicates in the training set inflate metrics and leak into validation.

In [4]:
print(f'Before dedup: {len(df)} rows')

# Identify duplicates with conflicting aspect labels
conflict = (
    df.groupby('review_text')['aspects']
    .nunique()
    .reset_index()
    .query('aspects > 1')
)
print(f'Duplicate texts with CONFLICTING labels: {len(conflict)}')

# For conflicting duplicates, keep the row with the most aspects annotated (more informative)
df['_n_aspects'] = df['aspects_list'].apply(len)
df = (
    df.sort_values('_n_aspects', ascending=False)
      .drop_duplicates(subset='review_text', keep='first')
      .drop(columns='_n_aspects')
      .reset_index(drop=True)
)

print(f'After dedup:  {len(df)} rows  (removed {1971 - len(df)})')

Before dedup: 1971 rows
Duplicate texts with CONFLICTING labels: 0
After dedup:  1883 rows  (removed 88)


---
## Step 3 — Normalize `business_category`
**Why:** 67 unique values — a mix of English and Arabic, many with <5 reviews.  
Grouping into ~8 macro-categories gives the model a meaningful domain signal.

In [5]:
RESTAURANT_KEYWORDS = ['مطعم', 'مقهى', 'كافيه', 'بيتزا', 'كريب', 'مشوية',
                       'سورية', 'برجر', 'شاورما', 'مأكولات', 'restaurant',
                       'cafe', 'coffee', 'bakery']
HEALTH_KEYWORDS     = ['مستشفى', 'عيادة', 'طبي', 'صحة', 'hospital',
                       'clinic', 'pharmacy', 'dental', 'أسنان']
HOTEL_KEYWORDS      = ['فندق', 'hotel', 'resort', 'نزل']
BEAUTY_KEYWORDS     = ['صالون', 'تجميل', 'spa', 'beauty', 'نails', 'حلاقة']
RETAIL_KEYWORDS     = ['سوق', 'متجر', 'تجزئة', 'retail', 'store', 'shop',
                       'بقالة', 'supermarket']
KNOWN_EXACT = {
    'ecommerce':     'ecommerce',
    'transport':     'transport',
    'travel':        'travel',
    'entertainment': 'entertainment',
    'real_estate':   'real_estate',
    'food_delivery': 'food_delivery',
}

def map_category(cat):
    cat_str = str(cat).strip()
    # Exact match first
    if cat_str.lower() in KNOWN_EXACT:
        return KNOWN_EXACT[cat_str.lower()]
    cat_lower = cat_str.lower()
    if any(k in cat_lower for k in RESTAURANT_KEYWORDS): return 'restaurant'
    if any(k in cat_lower for k in HEALTH_KEYWORDS):     return 'healthcare'
    if any(k in cat_lower for k in HOTEL_KEYWORDS):      return 'hotel'
    if any(k in cat_lower for k in BEAUTY_KEYWORDS):     return 'beauty'
    if any(k in cat_lower for k in RETAIL_KEYWORDS):     return 'retail'
    return 'other'

df['macro_category'] = df['business_category'].apply(map_category)

print('Macro category distribution:')
print(df['macro_category'].value_counts())
print(f'\nRemaining "other": {(df["macro_category"]=="other").sum()} rows')

Macro category distribution:
macro_category
restaurant       639
healthcare       248
ecommerce        183
transport        118
travel           117
entertainment    112
other             97
real_estate       97
beauty            83
food_delivery     80
retail            62
hotel             47
Name: count, dtype: int64

Remaining "other": 97 rows


---
## Step 4 — Clean Arabic Review Text
**Why:** Reviews contain URLs, HTML entities, repeated characters (تمامممم),  
extra whitespace, emojis, and mixed-script noise. A lightweight normalizer  
reduces vocabulary size without losing semantics.

In [6]:
def normalize_arabic(text: str) -> str:
    """
    Lightweight Arabic text normalizer.
    Does NOT remove emojis (they carry sentiment signal).
    Does NOT stem/lemmatize (handled by the pretrained tokenizer).
    """
    if not isinstance(text, str):
        return ''

    # 1. Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # 2. Remove HTML entities
    text = re.sub(r'&[a-zA-Z]+;', ' ', text)

    # 3. Normalize Arabic Alef variants → plain alef
    text = re.sub(r'[أإآ]', 'ا', text)

    # 4. Normalize teh marbuta
    text = re.sub(r'ة', 'ه', text)

    # 5. Remove tatweel (Arabic kashida / elongation character)
    text = re.sub(r'ـ', '', text)

    # 6. Reduce repeated characters: تمامممم → تمامم (keep max 2)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # 7. Remove diacritics (tashkeel)
    text = re.sub(r'[\u0610-\u061A\u064B-\u065F]', '', text)

    # 8. Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


df['review_text_clean'] = df['review_text'].apply(normalize_arabic)

# Verify on a few samples
samples = df[['review_text', 'review_text_clean']].head(5)
for _, row in samples.iterrows():
    print('ORIGINAL:', row['review_text'][:120])
    print('CLEANED: ', row['review_text_clean'][:120])
    print()

ORIGINAL: الحمام حجمه صغير وكل يوم بيرفعو السعر   والخبز بارد جدا    ولا يوجد شبكه        الدفع كاش ؟    والموظفين يطلبو بخشيش  ور
CLEANED:  الحمام حجمه صغير وكل يوم بيرفعو السعر والخبز بارد جدا ولا يوجد شبكه الدفع كاش ؟ والموظفين يطلبو بخشيش وريحه لحوم بالصاله

ORIGINAL: للأسف تجربتي كانت مخيبة للآمال. جودة الأكل أقل من المتوقع، والطعم عادي جدًا مقارنة بالسعر. الخدمة بطيئة وغير مهتمة ، واض
CLEANED:  للاسف تجربتي كانت مخيبه للامال. جوده الاكل اقل من المتوقع، والطعم عادي جدا مقارنه بالسعر. الخدمه بطيئه وغير مهتمه ، واضط

ORIGINAL: فندق هوليدي اي اكسبيرس الموقع ممتاز بجانب  فندق النبيله بشارع الدول العربية ومن خلال تجربتنا  التعامل مثالي والضيافة صحن
CLEANED:  فندق هوليدي اي اكسبيرس الموقع ممتاز بجانب فندق النبيله بشارع الدول العربيه ومن خلال تجربتنا التعامل مثالي والضيافه صحن ف

ORIGINAL: مطعم فطور غداء عشاء
تم تجربة الفطور
+ طريقة تقديم وطبخ مختلفة تستاهل التجربة.
+ تنوع كبير في الاصناف
- لا توجد مواقف

جو
CLEANED:  مطعم فطور غداء عشاء تم تجربه الفطور + طريقه تقديم وطبخ مختلفه تستاهل ا

> **Note on normalization depth:**  
> - We intentionally keep emojis and punctuation — they carry sentiment signal in noisy reviews  
> - We do NOT apply stemming/lemmatization — the pretrained AraBERT/CAMeLBERT tokenizer handles subword decomposition  
> - Alef normalization reduces vocabulary fragmentation (أ/إ/آ/ا are the same word)

---
## Step 5 — Handle Class Imbalance
**Why:**
- `neutral` is only 4.5% of sentiment labels
- `none` and `delivery` are rare aspects
- Ignoring this → model never predicts neutral

In [7]:
# --- Sentiment class weights (for loss function) ---
all_sentiments = [
    sent
    for d in df['sentiments_dict']
    for sent in d.values()
]
sent_series = pd.Series(all_sentiments)
sent_counts = sent_series.value_counts()
total = len(all_sentiments)

print('=== Sentiment label counts ===')
print(sent_counts)

# Inverse frequency weights
sent_weights = {label: total / (len(SENTIMENT_CLASSES) * count)
                for label, count in sent_counts.items()}
print('\nSentiment class weights (use in CrossEntropyLoss):')
for k, v in sorted(sent_weights.items()):
    print(f'  {k}: {v:.3f}')

# Convert to tensor-ready list (order: positive, negative, neutral)
weight_tensor = [sent_weights[s] for s in SENTIMENT_CLASSES]
print(f'\nWeight tensor (positive, negative, neutral): {[round(w,3) for w in weight_tensor]}')

# --- Aspect label counts ---
all_aspects = [a for lst in df['aspects_list'] for a in lst]
aspect_counts = pd.Series(all_aspects).value_counts()
print('\n=== Aspect mention counts ===')
print(aspect_counts)

=== Sentiment label counts ===
positive    1565
negative    1536
neutral      144
Name: count, dtype: int64

Sentiment class weights (use in CrossEntropyLoss):
  negative: 0.704
  neutral: 7.512
  positive: 0.691

Weight tensor (positive, negative, neutral): [0.691, 0.704, 7.512]

=== Aspect mention counts ===
service           988
food              454
app_experience    451
ambiance          378
price             354
general           220
cleanliness       185
delivery          161
none               54
Name: count, dtype: int64


---
## Step 6 — Create Training Formats

### Format A — Flattened (one row per aspect-sentiment pair)
Best for: **fine-tuning a sequence classifier** (e.g., AraBERT with aspect as input feature)  
Input: `[CLS] review_text [SEP] aspect [SEP]`  
Label: sentiment class (0=positive, 1=negative, 2=neutral)

In [8]:
SENTIMENT_TO_ID = {'positive': 0, 'negative': 1, 'neutral': 2}
ASPECT_TO_ID    = {a: i for i, a in enumerate(ASPECT_TAXONOMY)}

flat_records = []
for _, row in df.iterrows():
    for asp, sent in row['sentiments_dict'].items():
        flat_records.append({
            'review_id':      row['review_id'],
            'review_text':    row['review_text_clean'],
            'aspect':         asp,
            'aspect_id':      ASPECT_TO_ID.get(asp, -1),
            'sentiment':      sent,
            'sentiment_id':   SENTIMENT_TO_ID[sent],
            'star_rating':    row['star_rating'],
            'platform':       row['platform'],
            'macro_category': row['macro_category'],
            # Pre-formatted input string for the tokenizer:
            'model_input':    f"{row['review_text_clean']} [ASPECT] {asp}",
        })

flat_df = pd.DataFrame(flat_records)
print(f'Format A — flattened pairs: {len(flat_df)} rows')
print(flat_df[['review_text','aspect','sentiment','model_input']].head(4).to_string())

Format A — flattened pairs: 3245 rows
                                                                                                                                                review_text          aspect sentiment                                                                                                                                                                       model_input
0  الحمام حجمه صغير وكل يوم بيرفعو السعر والخبز بارد جدا ولا يوجد شبكه الدفع كاش ؟ والموظفين يطلبو بخشيش وريحه لحوم بالصاله ولا يوجد تهويه مايستاهل نجمه ❌️     cleanliness  negative     الحمام حجمه صغير وكل يوم بيرفعو السعر والخبز بارد جدا ولا يوجد شبكه الدفع كاش ؟ والموظفين يطلبو بخشيش وريحه لحوم بالصاله ولا يوجد تهويه مايستاهل نجمه ❌️ [ASPECT] cleanliness
1  الحمام حجمه صغير وكل يوم بيرفعو السعر والخبز بارد جدا ولا يوجد شبكه الدفع كاش ؟ والموظفين يطلبو بخشيش وريحه لحوم بالصاله ولا يوجد تهويه مايستاهل نجمه ❌️           price  negative           الحمام حجمه صغير وكل يوم بيرفعو السعر والخبز بارد 

### Format B — Per-review multi-label
Best for: **multi-task / pipeline approach** — first detect aspects, then classify sentiment  
Each row = one review with binary aspect presence vector + per-aspect sentiment labels

In [9]:
multi_records = []
for _, row in df.iterrows():
    # Binary vector: which aspects are present?
    aspect_binary = {f'asp_{a}': int(a in row['aspects_list']) for a in ASPECT_TAXONOMY}

    # Sentiment per aspect (NaN if aspect not mentioned)
    aspect_sent = {f'sent_{a}': row['sentiments_dict'].get(a, None) for a in ASPECT_TAXONOMY}
    aspect_sent_id = {f'sent_id_{a}': SENTIMENT_TO_ID.get(row['sentiments_dict'].get(a, ''), -1)
                      for a in ASPECT_TAXONOMY}

    record = {
        'review_id':      row['review_id'],
        'review_text':    row['review_text_clean'],
        'star_rating':    row['star_rating'],
        'platform':       row['platform'],
        'macro_category': row['macro_category'],
        'n_aspects':      len(row['aspects_list']),
    }
    record.update(aspect_binary)
    record.update(aspect_sent)
    record.update(aspect_sent_id)
    multi_records.append(record)

multi_df = pd.DataFrame(multi_records)
print(f'Format B — per-review multi-label: {len(multi_df)} rows')

# Show aspect presence columns
asp_cols = [c for c in multi_df.columns if c.startswith('asp_')]
print('\nAspect presence matrix (first 5 rows):')
print(multi_df[['review_text'] + asp_cols].head())

Format B — per-review multi-label: 1883 rows

Aspect presence matrix (first 5 rows):
                                         review_text  asp_food  asp_service  \
0  الحمام حجمه صغير وكل يوم بيرفعو السعر والخبز ب...         1            1   
1  للاسف تجربتي كانت مخيبه للامال. جوده الاكل اقل...         1            1   
2  فندق هوليدي اي اكسبيرس الموقع ممتاز بجانب فندق...         1            1   
3  مطعم فطور غداء عشاء تم تجربه الفطور + طريقه تق...         1            1   
4  طلبنا ٢ ايس لاتيه ده اسوء حاجه شربتها في حياتي...         1            1   

   asp_price  asp_cleanliness  asp_delivery  asp_ambiance  asp_app_experience  \
0          1                1             0             1                   1   
1          1                1             0             1                   0   
2          1                1             0             1                   0   
3          1                1             0             1                   0   
4          1                1      

---
## Step 7 — Train / Validation Split
**Rules:**
- Stratify by a combined key to preserve aspect + sentiment distribution
- 80/20 split
- No data leakage: split on `review_id` level (not on flattened pairs)

In [10]:
# Stratify on (platform × dominant_sentiment) — captures domain + label balance
from collections import Counter

def dominant_sentiment(d):
    if not d: return 'positive'
    return Counter(d.values()).most_common(1)[0][0]

df['dominant_sent'] = df['sentiments_dict'].apply(dominant_sentiment)
df['strat_key']     = df['platform'] + '_' + df['dominant_sent']

train_ids, val_ids = train_test_split(
    df['review_id'],
    test_size=0.2,
    random_state=SEED,
    stratify=df['strat_key']
)

train_ids = set(train_ids)
val_ids   = set(val_ids)

# Apply split to both formats
flat_train = flat_df[flat_df['review_id'].isin(train_ids)].reset_index(drop=True)
flat_val   = flat_df[flat_df['review_id'].isin(val_ids)].reset_index(drop=True)
multi_train = multi_df[multi_df['review_id'].isin(train_ids)].reset_index(drop=True)
multi_val   = multi_df[multi_df['review_id'].isin(val_ids)].reset_index(drop=True)

print(f'Format A  — Train: {len(flat_train)} pairs | Val: {len(flat_val)} pairs')
print(f'Format B  — Train: {len(multi_train)} reviews | Val: {len(multi_val)} reviews')

# Verify sentiment balance is preserved
print('\nSentiment distribution:')
print('  Train:', flat_train['sentiment'].value_counts(normalize=True).round(3).to_dict())
print('  Val:  ', flat_val['sentiment'].value_counts(normalize=True).round(3).to_dict())

Format A  — Train: 2608 pairs | Val: 637 pairs
Format B  — Train: 1506 reviews | Val: 377 reviews

Sentiment distribution:
  Train: {'positive': 0.482, 'negative': 0.475, 'neutral': 0.043}
  Val:   {'positive': 0.484, 'negative': 0.466, 'neutral': 0.05}


---
## Step 8 — Save Preprocessed Files

In [11]:
# Format A
flat_train.to_csv('train_flat.csv', index=False)
flat_val.to_csv('val_flat.csv',   index=False)

# Format B
multi_train.to_csv('train_multi.csv', index=False)
multi_val.to_csv('val_multi.csv',   index=False)

# Full cleaned dataset (useful for cross-validation)
df_save = df.drop(columns=['aspects', 'aspect_sentiments', 'strat_key'])
df_save['aspects_list']    = df_save['aspects_list'].apply(json.dumps)
df_save['sentiments_dict'] = df_save['sentiments_dict'].apply(json.dumps)
df_save.to_csv('train_clean_full.csv', index=False)

# Save class weights for model training
weights_info = {
    'sentiment_weights': {s: round(sent_weights[s], 4) for s in SENTIMENT_CLASSES},
    'weight_tensor_order': SENTIMENT_CLASSES,
    'weight_tensor': [round(w, 4) for w in weight_tensor],
    'sentiment_to_id': SENTIMENT_TO_ID,
    'aspect_to_id': ASPECT_TO_ID,
    'id_to_sentiment': {v: k for k, v in SENTIMENT_TO_ID.items()},
    'id_to_aspect': {v: k for k, v in ASPECT_TO_ID.items()},
}
with open('label_config.json', 'w', encoding='utf-8') as f:
    json.dump(weights_info, f, indent=2, ensure_ascii=False)

print('Saved:')
print('  train_flat.csv     — flattened aspect-sentiment pairs for training')
print('  val_flat.csv       — validation pairs')
print('  train_multi.csv    — per-review multi-label format')
print('  val_multi.csv      — validation multi-label')
print('  train_clean_full.csv — full cleaned dataset')
print('  label_config.json  — class weights, label mappings')

Saved:
  train_flat.csv     — flattened aspect-sentiment pairs for training
  val_flat.csv       — validation pairs
  train_multi.csv    — per-review multi-label format
  val_multi.csv      — validation multi-label
  train_clean_full.csv — full cleaned dataset
  label_config.json  — class weights, label mappings


---
## Step 9 — Quick Sanity Checks

In [12]:
print('=== Final shapes ===')
print(f'train_flat:  {flat_train.shape}')
print(f'val_flat:    {flat_val.shape}')
print(f'train_multi: {multi_train.shape}')
print(f'val_multi:   {multi_val.shape}')

print('\n=== No review leaks between train and val ===')
leak = set(flat_train['review_id']) & set(flat_val['review_id'])
print(f'Overlapping review_ids: {len(leak)}  (should be 0)')

print('\n=== model_input sample (Format A) ===')
for inp in flat_train['model_input'].head(3):
    print(' ', inp[:120])

print('\n=== label_config.json preview ===')
print(json.dumps(weights_info, indent=2, ensure_ascii=False))

=== Final shapes ===
train_flat:  (2608, 10)
val_flat:    (637, 10)
train_multi: (1506, 33)
val_multi:   (377, 33)

=== No review leaks between train and val ===
Overlapping review_ids: 0  (should be 0)

=== model_input sample (Format A) ===
  الحمام حجمه صغير وكل يوم بيرفعو السعر والخبز بارد جدا ولا يوجد شبكه الدفع كاش ؟ والموظفين يطلبو بخشيش وريحه لحوم بالصاله
  الحمام حجمه صغير وكل يوم بيرفعو السعر والخبز بارد جدا ولا يوجد شبكه الدفع كاش ؟ والموظفين يطلبو بخشيش وريحه لحوم بالصاله
  الحمام حجمه صغير وكل يوم بيرفعو السعر والخبز بارد جدا ولا يوجد شبكه الدفع كاش ؟ والموظفين يطلبو بخشيش وريحه لحوم بالصاله

=== label_config.json preview ===
{
  "sentiment_weights": {
    "positive": 0.6912,
    "negative": 0.7042,
    "neutral": 7.5116
  },
  "weight_tensor_order": [
    "positive",
    "negative",
    "neutral"
  ],
  "weight_tensor": [
    0.6912,
    0.7042,
    7.5116
  ],
  "sentiment_to_id": {
    "positive": 0,
    "negative": 1,
    "neutral": 2
  },
  "aspect_to_id": {
    "food"

---
## Summary

| Step | What was done | Why |
|---|---|---|
| Parse | `ast.literal_eval` on aspects/sentiments | They're stored as strings, not Python objects |
| Dedup | Removed 88 duplicate texts (kept most-annotated) | Prevents eval leakage and metric inflation |
| Category | Mapped 67 noisy categories → 8 macro-categories | Usable domain feature for the model |
| Text clean | Alef normalization, tatweel, diacritics, repeated chars | Reduces vocab noise without losing semantics |
| Class weights | Computed inverse-frequency weights for 3 sentiment classes | `neutral` is 4.5% — needs upweighting in loss |
| Format A | Flattened: 1 row per (review, aspect) pair with `[ASPECT]` tag | For sequence classification fine-tuning |
| Format B | 1 row per review with binary aspect presence + sentiment vectors | For multi-task / pipeline approach |
| Split | Stratified 80/20 on (platform × dominant_sentiment) | Preserves domain and label balance |

**Next step → Modeling:**  
Use `train_flat.csv` with `model_input` column as input to AraBERT / CAMeLBERT,  
`sentiment_id` as the target, and the `weight_tensor` from `label_config.json` in `nn.CrossEntropyLoss(weight=...)`.